# 04 · Recomendar un producto de un catálogo de 500

**Objetivo:** llega un prompt de un cliente ("busco algo para correr por monte, me tuerzo
los tobillos") → devolver **un producto** de mi catálogo.

## Aviso, por delante

Los notebooks 01–03 traen cifras medidas. **Este no trae ninguna**, y es a propósito: la
precisión de un recomendador depende de *tu* catálogo y de *tus* prompts, no de los míos.
Lo que trae es el **banco de pruebas y las tres variantes montadas**, para que los números
los saques tú antes de construir nada encima.

El catálogo y el banco que vienen dentro son **demo**. Sus números no significan nada.

## Las tres variantes

| | quién elige | opciones que ve Laya | ¿mide? |
|---|---|---|---|
| **V1 · solo retrieval** | coseno de embeddings | 0 (Laya no entra) | la baseline que hay que batir |
| **V2 · retrieval → Laya** | Laya, entre los `k` mejores | `k` (8) | ¿aporta Laya sobre V1? |
| **V3 · facetas → filtro** | tu código, con facetas de Laya | ≤10 categorías | ¿gana el determinismo? |

**V1 es la baseline y no es un hombre de paja.** Si V2 no la bate, Laya está costando 25 ms
por nada y la respuesta honesta es quedarse en V1.

## Por qué no se puede pasar el catálogo entero a un `choice`

Tres razones, en orden de dureza. Las tres se miden en la sección 1.

1. **Las opciones comparten un presupuesto de tokens.** `head_max_len` es 192 (inglés) / 256
   (multilingüe) para *todas* las opciones juntas. Con 500 productos toca ~4 tokens por
   producto: ni la descripción ni el slug completo entran. Medido aquí: con **20** opciones el
   margen top-2 ya cae a **0.0013**, y con 500 salta `ValueError`.
2. **La calibración de 11+ opciones viene rota** en el checkpoint inglés (`choice:11+` →
   temperatura 0.1006, que *afila* los logits ~10x). `laya` 0.3.20 la recorta a
   `TEMP_MIN=0.5` y avisa.
3. **`choice` es un softmax de conjunto cerrado.** No puede decir "ninguno de estos" — el
   notebook 03 lo midió: "Acuérdate de comprar pan" → `shop` con p=0.873. Un catálogo
   necesita esa puerta más que un enrutador.

Lo que sí funciona es el patrón **coarse-to-fine**, y `laya` ya lo trae en
`laya/shortlist.py`: retrieval baja 500 → `k`, y Laya decide entre `k`.

## 1 · El límite duro, medido

`Agent._encode_state` (agent.py:564) toma `head_max_len` del config del checkpoint, y
`build_sequence` (common.py:102-107) reparte ese presupuesto entre las opciones:

```
opt_budget = head_max_len - sum(len(o) for o in opt_ids)
if opt_budget < 16:
    per = max(4, (head_max_len - 16) // len(opt_ids))   # <- truncado a 4 tokens
    opt_ids = [o[:per] for o in opt_ids]
```

Cada opción es `[MASK]` + hasta 48 tokens de `"slug: descripción"`. Con `n` opciones el suelo
es **4 tokens por opción**: el `[MASK]` y tres tokens del slug. La descripción desaparece.

Y el aviso importante: **`head_max_len` no se comprueba.** El único error que salta es
`ValueError: question 'x' options exceed head_max_len=N`, y salta cuando los marcadores se
salen de `max_len` (agent.py:582-583). Si subes `max_len`, el error desaparece y la
degradación sigue ahí, en silencio.

La celda siguiente barre `n` y mide: tokens de entrada, si salta el error, y el margen top-2.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from laya import Router
from laya.common import TEMP_MAX, TEMP_MIN

MODELO = "multilingual"   # los prompts van en español; ver nota de calibración abajo
MAX_LEN = 2048

router = Router()
agent = router.load(MODELO)

print(f"checkpoint        : {MODELO}")
print(f"max_len config    : {agent.cfg.get('max_len')}")
print(f"head_max_len      : {agent.cfg.get('head_max_len')}   <- compartido por TODAS las opciones")
print(f"temperaturas      : {agent.temperature_raw}")
print(f"por nº de opciones: {agent.temperature_by_options_raw}")
print(f"clamp de la libreria: [{TEMP_MIN}, {TEMP_MAX}]")


En el multilingüe `temperature_by_options` sale **vacío** y las tres temperaturas son
`1.0`: este checkpoint no trae calibración por número de opciones. No es que 11+ esté roto —
es que no hay ninguna, así que las `probabilities` son softmax crudo para cualquier `k`.

El que trae calibración (y el bucket roto) es el inglés. Compruébalo tú en una línea, si
quieres pagar la carga de 1.7 GB:

```python
warnings.simplefilter("always")
router.load("english").temperature_by_options_raw
# {'choice:3-5': 1.760, 'choice:6-10': 1.000, 'choice:11+': 0.1006, ...}
```

Dos cosas que sacar de ahí a la hora de elegir `k`:

- `choice:11+` = **0.1006**, por debajo de `TEMP_MIN=0.5` → recortado, con `RuntimeWarning`.
  De ahí el límite de ≤10 del README.
- `choice:6-10` = **1.0000**, o sea *sin escalar*. Un shortlist a `k=8` cae en ese bucket y
  te da softmax crudo. Bajar a `k=5` cae en `choice:3-5` = **1.76**, que suaviza.
  **`k` no es solo cuántos candidatos ves: cambia si la probabilidad está calibrada o no.**

In [ ]:
def catalogo_sintetico(n):
    # Etiquetas sin significado a propósito: aquí se mide el presupuesto de tokens,
    # no la precision. Lo de los nombres con significado se mide en la seccion 9.
    return {f"producto-{i:04d}": "keyword, otra, tercera, cuarta" for i in range(n)}


def medir_limite(n, max_len=None):
    # max_len=None -> el del config del checkpoint (1024 en el multilingue)
    criteria = catalogo_sintetico(n)
    pregunta = {"p": {"type": "choice", "instructions": "Which product fits?",
                      "criteria": criteria}}
    fila = {"n": n, "max_len": max_len or agent.cfg.get("max_len")}
    try:
        r = router.predict("busco algo para correr por monte", pregunta, model=MODELO,
                           **({} if max_len is None else {"max_len": max_len}))
        p = sorted(r["answers"]["p"]["probabilities"].values(), reverse=True)
        fila.update(error=None, tokens=r["usage"]["input_tokens"],
                    p1=round(p[0], 4), margen=round(p[0] - p[1], 4))
    except ValueError as e:
        fila.update(error=str(e)[:52], tokens=None, p1=None, margen=None)
    return fila


# tokens reales que sobreviven por opcion, calculado igual que build_sequence
def tokens_por_opcion(n):
    hml = agent.cfg.get("head_max_len", 192)
    largos = [1 + len(agent.tok(" producto-0000: keyword, otra, tercera, cuarta",
                                add_special_tokens=False)["input_ids"])] * n
    if hml - sum(largos) < 16:
        return max(4, (hml - 16) // n)
    return largos[0]


filas = [medir_limite(n) for n in (5, 8, 10, 20, 50, 100, 200, 500)]
barrido = pd.DataFrame(filas)
barrido["tok/opcion"] = [tokens_por_opcion(n) for n in barrido["n"]]
barrido


Y ahora las mismas `n` subiendo `max_len`, para ver que el `ValueError` **no** protege
de nada — se va, y la degradación se queda:

In [ ]:
pd.DataFrame([medir_limite(n, ml) for ml in (MAX_LEN, 8192) for n in (200, 500)])


Esto es lo que sale en esta máquina, con el multilingüe y `max_len` por defecto (1024).
Es la única sección del notebook con cifras propias, porque las etiquetas son sintéticas y no
dependen de tu catálogo:

| n | tok entrada | tok/opción | p1 | margen |
|---|---|---|---|---|
| 5 | 93 | 15 | 0.298 | 0.085 |
| 8 | 138 | 15 | 0.249 | 0.086 |
| 10 | 168 | 15 | 0.270 | 0.137 |
| 20 | 258 | 12 | 0.107 | **0.0013** |
| 50 | **218** | 4 | 0.057 | 0.015 |
| 100 | 418 | 4 | 0.029 | 0.006 |
| 200 | 818 | 4 | 0.014 | 0.0006 |
| 500 | `ValueError` | 4 | — | — |

**Tres cosas, y las tres importan más que el límite de 10 del README:**

1. **Los tokens de entrada BAJAN de n=20 a n=50** (258 → 218). Es contraintuitivo y es la
   huella de la truncación: a partir de ahí cada opción cabe en 4 tokens y el total deja de
   depender de lo que escribiste. Más productos, menos texto por producto.
2. **`head_max_len=256` no se respeta.** Con n=200 la cabecera ocupa 800 tokens. El único
   guardia es que los marcadores no se salgan de `max_len`, y por eso el `ValueError` de n=500
   desaparece en cuanto subes `max_len` a 2048 (celda siguiente: `p1=0.016`, `margen=0.009`,
   sin error y sin valor). **No uses ese error como comprobación de cordura.**
3. **El fallo a escala es dilución, no sobreconfianza.** Con **20** opciones el margen top-2
   ya está en **0.0013**: muerto. Ojo al contraste con el 03, donde fuera de dominio y con
   pocas etiquetas salía *picudo y equivocado* (p=0.873). Son dos formas distintas de que
   `choice` no sirva: **pocas opciones → confiado y falso; muchas → plano e inútil.** Ninguna
   soporta un umbral.

**Aviso sobre estas cifras:** las 500 etiquetas son `producto-0000` con la misma descripción,
así que `p1` y `margen` miden dilución, no discriminación. Con productos de verdad el margen
sería mejor; la truncación a 4 tokens es idéntica.

## 2 · El catálogo

Las tres reglas del notebook 03 aplican igual, y aquí una de ellas pega **el doble**:

**El slug es señal dos veces.** `render_options` (common.py:56) construye cada opción como
`"slug: descripción"`. Ese mismo string es el que embebe el retrieval (shortlist.py,
`_option_texts`) *y* el que ve la cabeza de decisión. Un `sku_44712` no significa nada en
ninguna de las dos etapas. En el 03 el mismo banco daba 2/7 con etiquetas opacas y 7/7 con
palabras con significado; aquí el coste se paga dos veces.

**Descripciones = keywords, sin prosa.** Con 4 tokens por opción en el peor caso, la prosa
es presupuesto quemado. Nada de "Zapatilla técnica diseñada para...".

**Sin categoría `otros`.** Es un atractor: costaba 2 aciertos de 7.

⚠️ **Sustituye `CATALOGO` por el tuyo.** El de abajo es demo: 36 productos, 6 categorías,
vocabulario bien separado entre categorías y confundible dentro de cada una — que es
exactamente el reparto difícil de un catálogo real.

In [ ]:
# ⚠️ DEMO. Sustituyelo por tu catalogo real.
# 'kw' = keywords separadas por comas, sin prosa. 'sku' = slug legible, no codigo interno.
CATALOGO = [
    {"sku": "zapatilla-trail-mujer", "cat": "calzado-running", "precio": 129, "stock": 14, "nivel": "medio",
     "kw": "trail, montana, barro, mujer, drop bajo, agarre, tobillo"},
    {"sku": "zapatilla-trail-hombre", "cat": "calzado-running", "precio": 129, "stock": 9, "nivel": "medio",
     "kw": "trail, montana, barro, hombre, drop bajo, agarre, tobillo"},
    {"sku": "zapatilla-asfalto-neutra", "cat": "calzado-running", "precio": 145, "stock": 22, "nivel": "medio",
     "kw": "asfalto, ciudad, neutra, amortiguacion, maraton"},
    {"sku": "zapatilla-asfalto-pronador", "cat": "calzado-running", "precio": 139, "stock": 6, "nivel": "medio",
     "kw": "asfalto, pronador, soporte, estabilidad, plantilla"},
    {"sku": "zapatilla-iniciacion", "cat": "calzado-running", "precio": 69, "stock": 40, "nivel": "principiante",
     "kw": "principiante, empezar, primeras carreras, comoda, barata, 5k"},
    {"sku": "zapatilla-competicion-placa", "cat": "calzado-running", "precio": 249, "stock": 3, "nivel": "avanzado",
     "kw": "placa carbono, competicion, ligera, 10k, media maraton"},

    {"sku": "mochila-hidratacion-5l", "cat": "mochila", "precio": 89, "stock": 18, "nivel": "medio",
     "kw": "5 litros, hidratacion, chaleco, correr, agua, ligera"},
    {"sku": "mochila-senderismo-25l", "cat": "mochila", "precio": 75, "stock": 25, "nivel": "principiante",
     "kw": "25 litros, senderismo, un dia, dorsal ventilado, excursion"},
    {"sku": "mochila-travesia-60l", "cat": "mochila", "precio": 189, "stock": 7, "nivel": "avanzado",
     "kw": "60 litros, travesia, varios dias, carga, arnes, camino"},
    {"sku": "mochila-ataque-18l", "cat": "mochila", "precio": 99, "stock": 11, "nivel": "avanzado",
     "kw": "18 litros, ataque, alpinismo, porta piolet, cuerda"},
    {"sku": "mochila-ciudad-portatil", "cat": "mochila", "precio": 59, "stock": 33, "nivel": "principiante",
     "kw": "ciudad, portatil, 15 pulgadas, diario, acolchado"},
    {"sku": "mochila-nino-12l", "cat": "mochila", "precio": 39, "stock": 20, "nivel": "principiante",
     "kw": "nino, infantil, 12 litros, colegio, excursion"},

    {"sku": "chaqueta-goretex-3capas", "cat": "chaqueta-impermeable", "precio": 279, "stock": 8, "nivel": "avanzado",
     "kw": "Gore-Tex, 3 capas, impermeable, montana, lluvia fuerte, dias enteros"},
    {"sku": "chaqueta-cortavientos-ligera", "cat": "chaqueta-impermeable", "precio": 65, "stock": 30, "nivel": "principiante",
     "kw": "cortavientos, ligera, plegable, viento, correr"},
    {"sku": "chaqueta-plumas-800", "cat": "chaqueta-impermeable", "precio": 229, "stock": 5, "nivel": "avanzado",
     "kw": "plumas, 800 fill, frio seco, compresible, invierno"},
    {"sku": "chaqueta-primaloft-sintetica", "cat": "chaqueta-impermeable", "precio": 149, "stock": 12, "nivel": "medio",
     "kw": "sintetica, Primaloft, humedo, aislante, secado rapido"},
    {"sku": "chaqueta-softshell-transpirable", "cat": "chaqueta-impermeable", "precio": 119, "stock": 16, "nivel": "medio",
     "kw": "softshell, transpirable, elastica, entretiempo"},
    {"sku": "poncho-impermeable-mochila", "cat": "chaqueta-impermeable", "precio": 25, "stock": 45, "nivel": "principiante",
     "kw": "poncho, cubre mochila, emergencia, barato, lluvia"},

    {"sku": "saco-verano-10c", "cat": "saco-dormir", "precio": 55, "stock": 21, "nivel": "principiante",
     "kw": "verano, 10 grados, ligero, refugio, agosto, calor"},
    {"sku": "saco-3estaciones-0c", "cat": "saco-dormir", "precio": 139, "stock": 13, "nivel": "medio",
     "kw": "tres estaciones, 0 grados, plumon, montana"},
    {"sku": "saco-invierno-15c", "cat": "saco-dormir", "precio": 259, "stock": 4, "nivel": "avanzado",
     "kw": "invierno, menos 15 grados, expedicion, plumon, nieve, enero"},
    {"sku": "saco-sintetico-humedo", "cat": "saco-dormir", "precio": 69, "stock": 19, "nivel": "principiante",
     "kw": "sintetico, humedo, lavable, barato, kayak"},
    {"sku": "saco-mujer-anatomico", "cat": "saco-dormir", "precio": 159, "stock": 9, "nivel": "medio",
     "kw": "mujer, anatomico, pies mas aislados, 0 grados"},
    {"sku": "saco-doble-pareja", "cat": "saco-dormir", "precio": 129, "stock": 6, "nivel": "principiante",
     "kw": "doble, pareja, dos personas, camping, familiar"},

    {"sku": "tienda-1p-ultraligera", "cat": "tienda-campana", "precio": 319, "stock": 5, "nivel": "avanzado",
     "kw": "1 persona, ultraligera, bikepacking, 900 gramos"},
    {"sku": "tienda-2p-3estaciones", "cat": "tienda-campana", "precio": 199, "stock": 14, "nivel": "medio",
     "kw": "2 personas, tres estaciones, doble techo, montana"},
    {"sku": "tienda-4p-familiar", "cat": "tienda-campana", "precio": 249, "stock": 10, "nivel": "principiante",
     "kw": "4 personas, familiar, camping, habitaciones, playa"},
    {"sku": "tienda-geodesica-viento", "cat": "tienda-campana", "precio": 549, "stock": 2, "nivel": "avanzado",
     "kw": "geodesica, viento fuerte, expedicion, nieve"},
    {"sku": "tienda-techo-coche", "cat": "tienda-campana", "precio": 899, "stock": 1, "nivel": "medio",
     "kw": "techo de coche, furgoneta, camper, plataforma"},
    {"sku": "hamaca-mosquitera-selva", "cat": "tienda-campana", "precio": 79, "stock": 17, "nivel": "medio",
     "kw": "hamaca, mosquitera, selva, arboles, tropical"},

    {"sku": "frontal-200lm-usb", "cat": "frontal-linterna", "precio": 35, "stock": 38, "nivel": "principiante",
     "kw": "200 lumenes, USB, recargable, camping, ligero"},
    {"sku": "frontal-1000lm-trail", "cat": "frontal-linterna", "precio": 110, "stock": 12, "nivel": "avanzado",
     "kw": "1000 lumenes, trail nocturno, bateria externa, correr de noche"},
    {"sku": "frontal-pilas-emergencia", "cat": "frontal-linterna", "precio": 12, "stock": 60, "nivel": "principiante",
     "kw": "pilas AAA, emergencia, barato, coche, repuesto"},
    {"sku": "frontal-roja-astronomia", "cat": "frontal-linterna", "precio": 29, "stock": 15, "nivel": "medio",
     "kw": "luz roja, astronomia, vision nocturna, observacion"},
    {"sku": "linterna-mano-recargable", "cat": "frontal-linterna", "precio": 49, "stock": 22, "nivel": "medio",
     "kw": "linterna de mano, recargable, zoom, potente"},
    {"sku": "farol-camping-difuso", "cat": "frontal-linterna", "precio": 42, "stock": 18, "nivel": "principiante",
     "kw": "farol, difuso, mesa, tienda, ambiente"},
]

POR_SKU = {p["sku"]: p for p in CATALOGO}
CRITERIA = {p["sku"]: p["kw"] for p in CATALOGO}          # lo que ve Laya / el retrieval

# ⚠️ Categorias: maximo 10, y nombradas con palabras, no con codigos de tu ERP.
CATEGORIAS = {
    "calzado-running": "zapatillas, correr, running, trail, asfalto, pisada",
    "mochila": "mochila, litros, cargar, espalda, dorsal",
    "chaqueta-impermeable": "chaqueta, impermeable, lluvia, viento, abrigo, capa",
    "saco-dormir": "saco de dormir, dormir, grados, plumon, refugio",
    "tienda-campana": "tienda de campana, acampar, personas, camping, techo",
    "frontal-linterna": "frontal, linterna, luz, lumenes, iluminar, noche",
}

assert len(CATEGORIAS) <= 10, "11+ opciones: sin calibracion fiable en ningun checkpoint"
assert set(CATEGORIAS) == {p["cat"] for p in CATALOGO}, "categoria sin productos o al reves"
print(f"{len(CATALOGO)} productos, {len(CATEGORIAS)} categorias")
print(f"ejemplo de opcion tal como la ve el modelo: {CATALOGO[0]['sku']}: {CATALOGO[0]['kw']}")


## 3 · El banco de pruebas — esto es lo que tienes que rellenar

Es la única parte del notebook que no puedo escribir por ti, y la única que decide si el
resto sirve. **Mínimo 50 prompts reales**, tal como los escriben tus clientes: con faltas,
de tres palabras, sin nombrar el producto.

Tres clases de fila, y necesitas las tres:

| clase | qué es | para qué mide |
|---|---|---|
| `claro` | hay un producto correcto y solo uno | precisión (acc@1, acc@3) |
| `ambiguo` | la categoría está clara, el producto no | si el sistema sabe dudar |
| `fuera` | ningún producto del catálogo sirve | la puerta del no-match (sección 8) |

Si solo pones filas `claro`, vas a medir 90% y desplegar algo que recomienda zapatillas a
quien pregunta por un abogado. El notebook 03 se llevó ese golpe.

⚠️ `BANCO_DEMO` son 14 filas limpias y por tanto **optimistas**. Rellena `BANCO_REAL` y
tendrá prioridad automáticamente.

In [ ]:
# ⚠️ RELLENA ESTO. 50+ prompts reales tuyos. Se usa en cuanto tenga una fila.
BANCO_REAL = [
    # {"prompt": "...", "sku": "slug-de-tu-catalogo", "clase": "claro"},
    # {"prompt": "...", "sku": None, "clase": "ambiguo"},
    # {"prompt": "...", "sku": None, "clase": "fuera"},
]

BANCO_DEMO = [
    {"prompt": "quiero empezar a correr, algo comodo y que no sea muy caro",
     "sku": "zapatilla-iniciacion", "clase": "claro"},
    {"prompt": "zapatillas de trail para mujer, el suelo esta siempre mojado",
     "sku": "zapatilla-trail-mujer", "clase": "claro"},
    {"prompt": "voy a hacer el Camino, cinco dias, necesito llevar todo encima",
     "sku": "mochila-travesia-60l", "clase": "claro"},
    {"prompt": "algo para llevar agua mientras corro",
     "sku": "mochila-hidratacion-5l", "clase": "claro"},
    {"prompt": "me voy a Escocia, llueve todo el dia, quiero algo serio",
     "sku": "chaqueta-goretex-3capas", "clase": "claro"},
    {"prompt": "duermo en refugio en agosto y no quiero pasar calor",
     "sku": "saco-verano-10c", "clase": "claro"},
    {"prompt": "acampada en enero en la sierra, hace mucho frio y nieva",
     "sku": "saco-invierno-15c", "clase": "claro"},
    {"prompt": "vamos cuatro a un camping de playa en agosto",
     "sku": "tienda-4p-familiar", "clase": "claro"},
    {"prompt": "salgo a correr de noche por el monte y no veo nada",
     "sku": "frontal-1000lm-trail", "clase": "claro"},
    {"prompt": "una luz barata para dejar en el coche por si acaso",
     "sku": "frontal-pilas-emergencia", "clase": "claro"},
    {"prompt": "regalo para mi hermano, hace alpinismo", "sku": None, "clase": "ambiguo"},
    {"prompt": "algo para el frio", "sku": None, "clase": "ambiguo"},
    {"prompt": "busco un abogado laboralista en Madrid", "sku": None, "clase": "fuera"},
    {"prompt": "acuerdate de comprar pan", "sku": None, "clase": "fuera"},
]

BANCO = BANCO_REAL or BANCO_DEMO
ES_DEMO = not BANCO_REAL

for fila in BANCO:
    assert fila["clase"] in {"claro", "ambiguo", "fuera"}, fila
    assert fila["sku"] is None or fila["sku"] in POR_SKU, f"sku desconocido: {fila['sku']}"

CLAROS = [f for f in BANCO if f["clase"] == "claro"]
reparto = pd.Series([f["clase"] for f in BANCO]).value_counts().to_dict()
print(f"banco: {'DEMO — las cifras NO significan nada' if ES_DEMO else 'real'}")
print(f"{len(BANCO)} filas  {reparto}")
if len(CLAROS) < 30:
    print(f"\n⚠️  solo {len(CLAROS)} filas 'claro'. Con menos de 30, una diferencia de 2 "
          f"aciertos entre variantes es ruido: no decide nada.")


## 4 · V1 — solo retrieval, la baseline

`embed_fn_from_agent` reutiliza el encoder que ya tienes cargado y hace **mean-pooling**.
Aviso honesto: un encoder mmBERT/ModernBERT sin entrenamiento contrastivo es un retriever
mediocre. La propia docstring de `shortlist.py` lo dice: *"a dedicated bi-encoder passed as
`embed_fn` will usually shortlist better"*. Está aquí porque no añade dependencias y funciona
offline; la sección 10 trae la alternativa.

**Dos cosas que cuestan dinero en producción y no se ven en un notebook:**

1. `predict_shortlist` llama a `embed_fn` con `[query] + todas las opciones` **en cada
   petición**. Con 500 productos son 501 embeddings por prompt. Hay que cachear el catálogo:
   cambia una vez al día, no una vez por cliente.
2. El texto de cada opción es `"slug: keywords"` — exactamente lo que construye
   `render_options` (common.py:56). Lo replico a mano para que el ranking de V1 y la etapa de
   retrieval de V2 sean **idénticos** y la comparación signifique algo.

In [ ]:
from laya import embed_fn_from_agent, predict_shortlist

INSTRUCCION = "Which product fits this request?"
OPCION_TEXTO = {p["sku"]: f"{p['sku']}: {p['kw']}" for p in CATALOGO}   # == render_options


def con_cache(fn):
    # El catalogo se embebe una vez; solo la query es un miss por peticion.
    cache = {}

    def envuelto(textos):
        textos = list(textos)
        faltan = [t for t in dict.fromkeys(textos) if t not in cache]
        if faltan:
            for t, v in zip(faltan, fn(faltan)):
                cache[t] = v
        return np.stack([cache[t] for t in textos])

    envuelto.cache = cache
    return envuelto


embed_fn = con_cache(embed_fn_from_agent(agent))

t0 = time.perf_counter()
MATRIZ = embed_fn([OPCION_TEXTO[p["sku"]] for p in CATALOGO])
NORMAS = np.linalg.norm(MATRIZ, axis=1)
print(f"catalogo embebido: {MATRIZ.shape} en {time.perf_counter() - t0:.2f} s")


def ranking_retrieval(prompt, k=None):
    # _query_text de shortlist.py concatena instructions + estado. Mismo formato aqui.
    q = embed_fn([f"{INSTRUCCION}\n{prompt}"])[0]
    qn = np.linalg.norm(q)
    sims = np.zeros(len(CATALOGO)) if qn == 0 else (MATRIZ @ q) / np.maximum(NORMAS * qn, 1e-12)
    orden = np.argsort(-sims, kind="mergesort")[: k or len(CATALOGO)]
    return [(CATALOGO[int(i)]["sku"], float(sims[int(i)])) for i in orden]


def v1_retrieval(prompt):
    rank = ranking_retrieval(prompt, k=3)
    return {"sku": rank[0][0], "top3": [s for s, _ in rank],
            "coseno": round(rank[0][1], 4), "tokens": 0}


v1_retrieval(BANCO[0]["prompt"])


## 5 · V2 — retrieval → Laya

`predict_shortlist` (shortlist.py) hace las dos etapas en una llamada: embebe, se queda con
`k`, y lanza **una** pasada de Laya sobre ese subconjunto. No hay segunda pasada del modelo
de decisión.

**`k=8`, no el default de la librería.** `DEFAULT_SHORTLIST_K` es 20, que en el checkpoint
inglés cae en el bucket `choice:11+` — el de la temperatura recortada. Con 8 caes en
`choice:6-10`.

**La métrica que decide si V2 puede ganar es el `recall@k`**, no la precisión: si el producto
correcto no entra en el shortlist, Laya no tiene ninguna posibilidad de acertarlo. Techo de
V2 = recall@k. Se mide en la sección 7.

In [ ]:
K = 8   # NO el DEFAULT_SHORTLIST_K=20 de la libreria


def v2_shortlist(prompt, k=K, criteria=None):
    criteria = CRITERIA if criteria is None else criteria
    r = predict_shortlist(
        router, prompt,
        {"producto": {"type": "choice", "instructions": INSTRUCCION, "criteria": criteria}},
        embed_fn, k=k, model=MODELO, max_len=MAX_LEN,
    )
    a = r["answers"]["producto"]
    meta = r["shortlist"]["producto"]
    orden = sorted(a["probabilities"].items(), key=lambda kv: -kv[1])
    p1 = orden[0][1]
    p2 = orden[1][1] if len(orden) > 1 else 0.0
    return {"sku": orden[0][0], "top3": [s for s, _ in orden[:3]],
            "p1": round(p1, 4), "margen": round(p1 - p2, 4),
            "shortlist": meta["labels"],
            "coseno": round(meta["scores"][0], 4) if meta["scores"] else None,
            "tokens": r["usage"]["input_tokens"]}


v2_shortlist(BANCO[0]["prompt"])


## 6 · V3 — Laya saca facetas, tu código decide

La variante que montaría yo. Laya nunca ve los 500 productos: ve **6 categorías** y unas
cuantas preguntas `noul` concretas. Con eso, el filtro sobre el catálogo lo hace tu código,
con tus datos (precio, stock, nivel, talla, márgenes) y de forma auditable.

Por qué encaja con lo medido en 01–03:

- **≤10 opciones siempre.** El presupuesto de tokens deja de ser un problema.
- **`noul` concretos.** El 03 midió que los `noul` abstractos ("¿es de trabajo?") salen
  invertidos o pegados a 0, y los concretos ("threaten to cancel" → 0.879) funcionan.
  "¿pide algo barato?" es concreto: hay vocabulario en el texto que lo delata.
- **Todas las preguntas en una pasada.** Seis facetas cuestan lo mismo en latencia que una.
- **El "no hay producto" lo decides tú**, no un softmax.

Lo que V3 **no** arregla: `categoria` sigue siendo un `choice` de conjunto cerrado. "Busco un
abogado" va a caer en alguna categoría con confianza. La puerta del no-match sigue estando
fuera (sección 8).

⚠️ Este es el bloque que más hay que reescribir con tu negocio: las facetas que importan en
un catálogo de zapatillas no son las de uno de software.

In [ ]:
FACETAS = {
    "categoria": {"type": "choice",
                  "instructions": "¿De qué categoría de producto está hablando el cliente?",
                  "criteria": CATEGORIAS},
    "pide_barato": {"type": "noul",
                    "instructions": "¿El cliente pide algo barato, economico o de bajo precio?"},
    "es_principiante": {"type": "noul",
                        "instructions": "¿El cliente dice que empieza o que es principiante?"},
    "menciona_lluvia": {"type": "noul",
                        "instructions": "¿El cliente menciona lluvia, agua, suelo mojado o humedad?"},
    "menciona_frio": {"type": "noul",
                      "instructions": "¿El cliente menciona frio, invierno, nieve o bajo cero?"},
    "es_regalo": {"type": "noul",
                  "instructions": "¿El cliente compra esto como regalo para otra persona?"},
}
NOULS = [q for q in FACETAS if q != "categoria"]
UMBRAL_NOUL = 0.5

PISTAS_FACETA = {
    "menciona_lluvia": ("impermeable", "lluvia", "mojado", "humedo", "gore-tex", "poncho"),
    "menciona_frio": ("frio", "invierno", "nieve", "grados", "plumon", "aislante"),
}


def v3_facetas(prompt):
    r = router.predict(prompt, FACETAS, model=MODELO, max_len=MAX_LEN)
    a = r["answers"]
    cat_orden = sorted(a["categoria"]["probabilities"].items(), key=lambda kv: -kv[1])
    cat, p1 = cat_orden[0]
    facetas = {q: a[q]["noul"] for q in NOULS}

    # Filtro determinista: tus datos, tus reglas, auditable linea a linea.
    cand = [p for p in CATALOGO if p["cat"] == cat and p["stock"] > 0]
    caro = max((p["precio"] for p in cand), default=1) or 1

    def puntuar(p):
        s = 0.0
        if facetas["pide_barato"] > UMBRAL_NOUL:
            s += 1.0 - p["precio"] / caro
        if facetas["es_principiante"] > UMBRAL_NOUL and p["nivel"] == "principiante":
            s += 0.6
        for q, pistas in PISTAS_FACETA.items():
            if facetas[q] > UMBRAL_NOUL and any(k in p["kw"].lower() for k in pistas):
                s += 0.5
        if facetas["es_regalo"] > UMBRAL_NOUL and p["stock"] >= 5:
            s += 0.1
        return (s, p["stock"], -p["precio"])   # desempate estable

    rank = sorted(cand, key=puntuar, reverse=True)
    return {"sku": rank[0]["sku"] if rank else None,
            "top3": [p["sku"] for p in rank[:3]],
            "categoria": cat, "p1": round(p1, 4),
            "margen": round(p1 - cat_orden[1][1], 4),
            "facetas": {q: round(v, 3) for q, v in facetas.items()},
            "tokens": r["usage"]["input_tokens"]}


v3_facetas(BANCO[0]["prompt"])


### Antes de la comparativa: ¿funcionan las facetas?

Mismo control que la sección "lo que se midió roto" del 03. Si un `noul` sale invertido o
pegado a un extremo en **todas** las filas, no está midiendo nada y hay que quitarlo del
filtro — deja de ser señal y pasa a ser ruido con nombre bonito.

In [ ]:
filas = []
for f in BANCO:
    a = router.predict(f["prompt"], FACETAS, model=MODELO, max_len=MAX_LEN)["answers"]
    filas.append({"prompt": f["prompt"][:38], "clase": f["clase"],
                  "categoria": a["categoria"]["choice"],
                  "cat_p1": round(max(a["categoria"]["probabilities"].values()), 3),
                  **{q: round(a[q]["noul"], 3) for q in NOULS}})

facetas_df = pd.DataFrame(filas)
print("rango de cada noul sobre el banco (si min~=max, no discrimina):")
print(facetas_df[NOULS].agg(["min", "max", "std"]).round(3).T)
facetas_df


## 7 · La comparativa

Solo sobre las filas `claro` — son las únicas con respuesta correcta conocida.
`recall@k` es el techo de V2: si el correcto no entra en el shortlist, no hay nada que elegir.

In [ ]:
# El embedding de la query lo pagan las tres variantes igual, asi que se precalienta
# antes de cronometrar: 'ms' es el coste MARGINAL de cada variante, no el total.
t0 = time.perf_counter()
for f in CLAROS:
    embed_fn([f"{INSTRUCCION}\n{f['prompt']}"])
ms_query = (time.perf_counter() - t0) * 1000 / len(CLAROS)

reg = []
for f in CLAROS:
    for nombre, fn in (("V1 retrieval", v1_retrieval),
                       ("V2 shortlist", v2_shortlist),
                       ("V3 facetas", v3_facetas)):
        t0 = time.perf_counter()
        d = fn(f["prompt"])
        ms = (time.perf_counter() - t0) * 1000
        reg.append({"variante": nombre, "prompt": f["prompt"][:34],
                    "esperado": f["sku"], "predicho": d["sku"],
                    "ok1": d["sku"] == f["sku"], "ok3": f["sku"] in d["top3"],
                    "ms": round(ms, 1), "tokens": d.get("tokens", 0)})

detalle = pd.DataFrame(reg)

# recall@K de la etapa de retrieval. V1 y V2 la comparten, asi que es UN numero y no una
# columna por variante: es el techo de V2, lo que no entra no se puede elegir. Va DESPUES
# del cronometro a proposito; calcularlo antes calienta la cache y falsea el ms de V1.
recall_k = float(np.mean([f["sku"] in [s for s, _ in ranking_retrieval(f["prompt"], k=K)]
                          for f in CLAROS]))

resumen = detalle.groupby("variante").agg(
    aciertos=("ok1", "sum"), acc1=("ok1", "mean"), acc3=("ok3", "mean"),
    ms_p50=("ms", "median"), tokens=("tokens", "median"),
).round(3)
resumen["aciertos"] = resumen["aciertos"].astype(str) + f"/{len(CLAROS)}"

print(f"banco {'DEMO (cifras sin valor)' if ES_DEMO else 'real'} — {len(CLAROS)} filas claras")
print(f"recall@{K} del retrieval (techo de V2): {recall_k:.3f}")
print(f"embedding de la query: {ms_query:.1f} ms/prompt, igual para las tres (no en 'ms')")
resumen


**Cómo leer esto, y en qué orden:**

1. **`recall@8` de V2.** Si no llega a ~0.9, el problema es el retrieval, no Laya. Arregla
   primero las keywords o cambia de `embed_fn` (sección 10); ajustar Laya no va a servir.
2. **`acc1` de V2 vs `acc1` de V1.** Es *la* comparación del notebook. Si V2 no gana,
   **quédate en V1**: Laya está añadiendo 25 ms y una dependencia sin comprar nada. No es un
   resultado decepcionante, es el resultado.
3. **`acc1` de V2 vs `acc3` de V1.** Si V1 mete el correcto en su top-3 pero falla el top-1, y
   V2 lo rescata, ahí está el valor concreto de Laya: **reordenar**, que es lo que un
   coseno de mean-pooling hace peor.
4. **V3 contra las dos.** V3 pierde el matiz dentro de la categoría (no lee "800 fill"), pero
   gana precio, stock y nivel, que el retrieval no ve. Si V3 gana, tu problema no era
   semántico: era de filtros.
5. **`tokens`.** V3 gasta menos (6 categorías + 5 `noul` cortos) que V2 (8 productos con sus
   keywords). Con `max_len=2048` ninguna se acerca al techo; con el checkpoint inglés (512)
   sí importa.
6. **`ms` es coste marginal, no latencia total.** El embedding de la query se precalienta antes
   de cronometrar porque lo pagan las tres igual (se imprime aparte), y el catálogo va cacheado.
   Para la latencia de producción suma `ms_query` + tu tiempo de índice.

Y el detalle fila a fila, que es donde se ven los patrones:

In [ ]:
detalle[~detalle["ok1"]][["variante", "prompt", "esperado", "predicho"]].reset_index(drop=True)


## 8 · El no-match: lo único que ninguna variante arregla sola

Un enrutador de proyectos puede permitirse fallar aquí. **Un recomendador no**: el cliente que
pregunta por algo que no vendes es un caso frecuente, no un borde.

El 03 lo dejó medido: `choice` es un softmax de conjunto cerrado, reparte toda la masa entre
las opciones que le das, y fuera de dominio produce distribuciones **picudas y equivocadas**
("Acuérdate de comprar pan" → `shop`, p=0.873, *más* confiado que un mensaje real). Ningún
umbral sobre `p1` separa eso, y da igual que sea V2 o la `categoria` de V3.

Lo que sí es de **conjunto abierto** es el coseno del retrieval: no está normalizado contra
un conjunto de etiquetas, así que un prompt que no se parece a nada del catálogo puntúa bajo
contra todo. **Ahí es donde va la puerta.**

La celda siguiente contrasta las dos señales por clase de fila. No te fíes de mí: mira si se
separan **en tu catálogo**.

In [ ]:
filas = []
for f in BANCO:
    r1, r2, r3 = v1_retrieval(f["prompt"]), v2_shortlist(f["prompt"]), v3_facetas(f["prompt"])
    filas.append({"prompt": f["prompt"][:36], "clase": f["clase"],
                  "coseno": r1["coseno"], "laya_p1": r2["p1"], "laya_margen": r2["margen"],
                  "cat_p1": r3["p1"], "predicho": r2["sku"]})

nomatch = pd.DataFrame(filas)
print("por clase de fila (abierto vs cerrado):")
print(nomatch.groupby("clase")[["coseno", "laya_p1", "cat_p1"]]
      .agg(["min", "median", "max"]).round(3))
nomatch


Y el barrido de umbral, que es lo que de verdad decides. Un umbral solo sirve si rechaza
lo de fuera **sin** tirar lo de dentro:

In [ ]:
dentro = nomatch[nomatch["clase"] == "claro"]["coseno"]
fuera = nomatch[nomatch["clase"] == "fuera"]["coseno"]

if len(fuera) < 10:
    print(f"⚠️  solo {len(fuera)} filas 'fuera'. Con menos de 10 este barrido no decide nada:")
    print("    mete prompts reales de gente preguntando por lo que NO vendes.\n")

barrido_umbral = pd.DataFrame([
    {"umbral": round(t, 3),
     "rechaza fuera": f"{int((fuera < t).sum())}/{len(fuera)}",
     "tira claros": f"{int((dentro < t).sum())}/{len(dentro)}"}
    for t in np.quantile(nomatch["coseno"], np.linspace(0.05, 0.95, 10))
])
barrido_umbral


**Qué esperar, y qué hacer con cada caso:**

- **Se separan** (hay una fila con `rechaza fuera` alto y `tira claros` en 0): ya tienes la
  puerta. Va **antes** de Laya, y ahorra la pasada entera.
- **No se separan**: el mean-pooling no te da margen. Dos salidas, por orden de coste: un
  bi-encoder de verdad (sección 10), o una puerta que no sea semántica — que el cliente elija
  categoría en la UI, y el modelo solo decida dentro.
- **`laya_p1` alto en las filas `fuera`** es el resultado esperado, no un fallo del modelo. Es
  la propiedad del softmax cerrado. Si lo ves, la medición está bien hecha.

## 9 · Las ablaciones que hay que correr antes de creerse nada

Tres, en orden de cuánto movieron la aguja en el 03. Todas pasan por V2, así que miden el
pipeline completo, no el modelo desnudo.

In [ ]:
def ablacion(nombre, criteria, mapa=None, k=K):
    ok, toks, p1s = 0, 0, []
    for f in CLAROS:
        d = v2_shortlist(f["prompt"], k=k, criteria=criteria)
        ok += (mapa[d["sku"]] if mapa else d["sku"]) == f["sku"]
        toks = d["tokens"]
        p1s.append(d["p1"])
    print(f"{nombre:<46} {ok}/{len(CLAROS)}   tokens={toks:<5} "
          f"p1={min(p1s):.2f}-{max(p1s):.2f}")


print("A · k (cuantos candidatos ve Laya)")
for k in (3, 5, 8, 20, len(CATALOGO)):
    etiqueta = f"    k={k}" + ("  <- passthrough, Laya ve TODO" if k >= len(CATALOGO)
                               else "  <- el default del notebook" if k == K
                               else "  <- DEFAULT_SHORTLIST_K de la libreria" if k == 20 else "")
    ablacion(etiqueta, CRITERIA, k=k)


In [ ]:
print("B · keywords vs prosa (misma informacion, mas masa semantica)")
PROSA = {p["sku"]: f"Producto de la categoria {p['cat']}, pensado para {p['kw']}."
         for p in CATALOGO}
ablacion("    keywords  <- el default", CRITERIA)
ablacion("    prosa", PROSA)

print("\nC · slug legible vs codigo interno")
OPACO = {f"sku_{i:05d}": p["kw"] for i, p in enumerate(CATALOGO)}
MAPA_OPACO = {f"sku_{i:05d}": p["sku"] for i, p in enumerate(CATALOGO)}
ablacion("    slug legible  <- el default", CRITERIA)
ablacion("    sku_00000", OPACO, mapa=MAPA_OPACO)


**Qué esperar de cada una:**

- **A · `k`.** `k=3` recorta el recall (el correcto se queda fuera) y `k=36` deja a Laya
  repartiendo 256 tokens entre 36 opciones. El óptimo está en medio y **depende de tu
  recall@k**, no de un número que te pueda dar yo. Si `k=36` no es mucho peor que `k=8`, tu
  catálogo es demasiado pequeño para necesitar shortlist.
  En el catálogo demo **`k=5` gana a `k=8` y a `k=20`**, que no es lo que esperarías: con un
  retriever flojo, subir `k` mete más distractores que aciertos. Si te pasa lo mismo, no es que
  `k` pequeño sea mejor — es que **tu retrieval no está poniendo el correcto arriba**, y la
  respuesta está en la sección 10.
- **B · prosa.** En el 03 la prosa costaba 1 acierto y el doble de tokens. Aquí la prosa está
  generada mecánicamente; **el copy real de una ficha de producto es mucho peor** — párrafos
  de marketing que compiten con la señal y se comen el presupuesto.
- **C · el slug.** La que más movió en el 03 (2/7 → 7/7). Y aquí pega dos veces: el string
  `"slug: keywords"` es lo que embebe el retrieval **y** lo que ve la cabeza de decisión. Si
  tu catálogo vive con SKUs opacos, esta ablación te dice exactamente cuánto te cuesta — y
  el arreglo es una columna de slugs, no un modelo mejor.

## 10 · Si el retrieval es el cuello de botella

`embed_fn_from_agent` hace mean-pooling de un encoder que **no** se entrenó para retrieval.
Es gratis y offline, pero si el `recall@k` de la sección 7 sale bajo, ahí está tu techo y no
lo vas a subir tocando Laya.

**Medido en el catálogo demo de este notebook** (36 productos, 10 prompts — muestra pequeña,
tómalo como orden de magnitud): `recall@8 = 0.60` y **V1 acierta 0/10**. Con 4 de cada 10
productos correctos fuera del shortlist, V2 tiene el techo en 6/10 antes de empezar. El cuello
de botella del pipeline es el `embed_fn`, no el clasificador — y esto es un hallazgo sobre el
helper de la librería, no sobre tu catálogo.

La celda siguiente cambia solo el `embed_fn` y deja el resto igual, para que la comparación
sea limpia. Está apagada por defecto: añade dependencia y descarga pesos.

In [ ]:
USAR_BIENCODER = False   # ⚠️ instala sentence-transformers y descarga ~470 MB

if USAR_BIENCODER:
    from sentence_transformers import SentenceTransformer

    st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    embed_st = con_cache(lambda textos: np.asarray(
        st.encode(list(textos), convert_to_numpy=True, show_progress_bar=False)))

    aciertos = recall = 0
    for f in CLAROS:
        r = predict_shortlist(
            router, f["prompt"],
            {"producto": {"type": "choice", "instructions": INSTRUCCION, "criteria": CRITERIA}},
            embed_st, k=K, model=MODELO, max_len=MAX_LEN)
        aciertos += r["answers"]["producto"]["choice"] == f["sku"]
        recall += f["sku"] in r["shortlist"]["producto"]["labels"]
    print(f"bi-encoder multilingue:  acc1={aciertos}/{len(CLAROS)}  "
          f"recall@{K}={recall}/{len(CLAROS)}")
else:
    print("apagado. Pon USAR_BIENCODER=True si el recall@k de la seccion 7 te decepciono.")
    print(f"embeddings cacheados del encoder de Laya: {len(embed_fn.cache)} textos")


---

## Conclusión honesta

**Laya no puede ser tu recomendador.** Puede ser una de sus tres piezas, y cuál depende de
números que solo salen de tu catálogo:

```
prompt del cliente
   │
   ├─ 1. puerta de no-match por COSENO       ← conjunto abierto; Laya no sabe hacer esto
   ├─ 2. retrieval 500 -> k                  ← aquí está casi toda la precisión
   ├─ 3. Laya: reordenar los k  (V2)         ← 25 ms; solo si bate al top-1 de V2
   │      o Laya: facetas -> filtro (V3)     ← si tu problema es precio/stock/nivel
   └─ 4. margen estrecho -> preguntar o escalar a un LLM
```

**Los pasos 1 y 2 no son de Laya.** Si solo te llevas una cosa de este notebook: el trabajo
de un recomendador sobre 500 productos está en el retrieval y en los datos del catálogo, no
en el clasificador. Laya entra en el paso 3, sobre 8 candidatos, y ahí gana o no gana según
lo que mida la sección 7.

**Un LLM haría el paso 3 mejor.** Lee "800 fill", entiende "me tuerzo los tobillos" → tobillo
alto, razona sobre precio. Razones reales para Laya en un catálogo:

1. **Coste por petición.** Un recomendador se llama en cada visita, no en cada ticket. A
   volumen de e-commerce, 25 ms de CPU local contra una llamada de API es la diferencia
   entre gratis y una factura mensual.
2. **Probabilidades para hacer aritmética.** Puedes multiplicar por margen, por stock, por un
   prior de conversión. Un string no.
3. **Latencia predecible.** 25 ms sin red, sin rate limit, sin caída de proveedor en Black
   Friday.

**Lo que montaría en serio: cascada.** Puerta por coseno → V3 resuelve la mayoría con filtros
determinista → margen estrecho escala a un LLM → la respuesta del LLM se registra y es el
dataset del fine-tune. Es el mismo patrón del 03, aplicado a un catálogo.

### Orden recomendado

1. **Slugs legibles en el catálogo.** Antes de nada. Ablación C te dice lo que cuesta no
   tenerlos, y es la única mejora que no requiere tocar código de modelo.
2. **50 prompts reales en `BANCO_REAL`**, con sus filas `ambiguo` y `fuera`. Sin esto, todo
   lo demás son cifras de un catálogo de demo.
3. **Sección 7, y mira el `recall@k` primero.** Si es bajo, vete a la sección 10 y para.
4. **V1 vs V2.** Si V2 no gana, despliega V1 y ahórrate la dependencia.
5. **Sección 8.** La puerta de no-match. En un recomendador es requisito, no extra.
6. Solo entonces: V3 con tus facetas de negocio, y la cascada a un LLM.

### Lo que este notebook no cubre

- **Ranking comercial** (margen, stock, novedad, devoluciones). Es la mitad del valor de un
  recomendador real y no es un problema de modelos.
- **Historial del cliente.** Laya no tiene memoria; va como prior sobre las probabilidades,
  igual que en el 03 sección 4.
- **Catálogo que cambia.** Los embeddings se recalculan al cambiar un producto. `con_cache`
  de la sección 4 es un juguete: en producción eso es un índice con invalidación.
- **Varias categorías en un prompt** ("zapatillas y un frontal"). `choice` devuelve una.